In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from tqdm.notebook import tqdm  # Import tqdm specifically optimized for Jupyter

# 1. Load environment variables from your .env file
load_dotenv()
print("Connecting to PostgreSQL...")

# 2. Securely fetch the credentials
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# 3. Construct the database URL and create the engine
db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

# 4. Ask PostgreSQL exactly how many rows exist so the progress bar knows what 100% is
count_query = "SELECT COUNT(*) FROM public.model_features_v05;"
total_rows = pd.read_sql(count_query, engine).iloc[0, 0]

# 5. Fetch the data in chunks and update the progress bar
query = "SELECT * FROM public.model_features_v05;"
chunksize = 100000 
chunks = [] # We will store our data chunks here temporarily

print(f"Fetching {total_rows:,} engineered features from local database...")

# Create the progress bar
with tqdm(total=total_rows, desc="Downloading Data") as pbar:
    # Read the data in chunks
    for chunk in pd.read_sql(query, engine, chunksize=chunksize):
        chunks.append(chunk)      # Save the chunk
        pbar.update(len(chunk))   # Move the progress bar forward

# 6. Stitch all the chunks together into one massive DataFrame
df = pd.concat(chunks, ignore_index=True)

print(f"Success! Loaded {len(df):,} rows into local memory.")
display(df.head())

Connecting to PostgreSQL...
Fetching 5,732,926 engineered features from local database...


Success! Loaded 5,732,926 rows into local memory.


,MONTH,DAY_OF_WEEK,AIRLINE_ID,ORIGIN_AIRPORT_ID,DESTINATION_AIRPORT_ID,DISTANCE,DEPARTURE_BLOCK,AIRLINE_HISTORIC_DELAY,ROUTE_HISTORIC_DELAY_BY_AIRLINE,ORIGIN_PRECIPITATION,...,ORIGIN_CLOUD_COVER_LOW,DEST_PRECIPITATION,DEST_SNOWFALL,DEST_WIND_SPEED,DEST_WIND_GUSTS,DEST_CLOUD_COVER_LOW,PREVIOUS_FLIGHT_DELAYED,ORIGIN_HOURLY_AIRPORT_TRAFFIC,DEST_HOURLY_AIRPORT_TRAFFIC,IS_DELAYED
0,11,5,9,86.0,278.0,1024,Afternoon,10.58,14.33,0.0,...,0,0.0,0.0,9.7,18.4,0,0,37,14,1
1,11,1,2,177.0,23.0,1242,Morning,8.90,6.27,0.0,...,87,0.0,0.0,2.9,13.7,0,0,34,10,0
2,7,2,2,23.0,177.0,1242,Afternoon,8.90,8.15,0.0,...,2,0.0,0.0,5.4,14.0,98,0,9,28,0
3,7,5,2,23.0,177.0,1242,Evening,8.90,8.15,0.0,...,0,0.0,0.0,16.5,36.7,36,0,7,57,0
4,9,7,2,23.0,177.0,1242,Evening,8.90,8.15,0.0,...,0,0.0,0.0,10.5,29.2,0,0,2,48,0


In [2]:
from sklearn.model_selection import train_test_split

print("Starting preprocessing...")

# 1. Drop columns we don't need for the predictive model
# We drop ORIGIN_AIRPORT and DESTINATION_AIRPORT to keep the model fast, 
# relying on the ROUTE_HISTORIC_DELAY feature instead.
columns_to_drop = ['MONTH', 'DAY_OF_WEEK']
df_clean = df.drop(columns=columns_to_drop)

# 2. One-Hot Encode the categorical text columns (AIRLINE, DEPARTURE_BLOCK)
# This turns categories into binary numerical columns using pandas
df_encoded = pd.get_dummies(df_clean, columns=['DEPARTURE_BLOCK'], drop_first=True)

# 3. Separate your Features (X) from your Target (y)
X = df_encoded.drop(columns=['IS_DELAYED'])
y = df_encoded['IS_DELAYED']

# 4. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Preprocessing complete!")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Starting preprocessing...
Preprocessing complete!
Training features shape: (4586340, 22)
Testing features shape: (1146586, 22)


In [3]:
from xgboost import XGBClassifier
import time

print("Initializing XGBoost Classifier...")
# We use standard default parameters to establish a strong baseline
model = XGBClassifier(
    random_state=42, 
    eval_metric='logloss' # Tells the model how to measure its own errors during training
)

print("Training the model on 80% of the data. Please wait...")
start_time = time.time()

# This single line is where the actual math and "learning" happens
model.fit(X_train, y_train)

end_time = time.time()
print(f"Success! Training complete in {round(end_time - start_time, 2)} seconds.")

Initializing XGBoost Classifier...
Training the model on 80% of the data. Please wait...
Success! Training complete in 13.1 seconds.


In [4]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Evaluating model on the hidden test data...")

# 1. Ask the model to make predictions
y_pred = model.predict(X_test)

# 2. Print out the KPI metrics
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 3. Export the trained model to your hard drive
model_filename = 'xgboost_delay_model_v05_v00.joblib'
joblib.dump(model, model_filename)

print(f"\nBoom! Model fully trained, evaluated, and saved locally as: {model_filename}")

Evaluating model on the hidden test data...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.88      0.96      0.92    942296
           1       0.70      0.41      0.51    204290

    accuracy                           0.86   1146586
   macro avg       0.79      0.68      0.72   1146586
weighted avg       0.85      0.86      0.85   1146586

--- Confusion Matrix ---
[[906015  36281]
 [121469  82821]]

Boom! Model fully trained, evaluated, and saved locally as: xgboost_delay_model_v05_v00.joblib


In [5]:
from xgboost import XGBClassifier
import time

print("Calculating class imbalance...")
# Count the 0s (On Time) and divide by the 1s (Delayed)
# Based on your confusion matrix, this will be roughly 4.6
imbalance_ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"Imbalance Ratio: {round(imbalance_ratio, 2)} to 1")

print("\nInitializing Tuned XGBoost Classifier...")
model = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    scale_pos_weight=imbalance_ratio  # <-- The magic parameter
)

print("Training the model. Please wait...")
start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

print(f"Success! Training complete in {round(end_time - start_time, 2)} seconds.")

Calculating class imbalance...
Imbalance Ratio: 4.63 to 1

Initializing Tuned XGBoost Classifier...
Training the model. Please wait...
Success! Training complete in 13.08 seconds.


In [6]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Evaluating model on the hidden test data...")

# 1. Ask the model to make predictions
y_pred = model.predict(X_test)

# 2. Print out the KPI metrics
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 3. Export the trained model to your hard drive
model_filename = 'xgboost_delay_model_v05_v01.joblib'
joblib.dump(model, model_filename)

print(f"\nBoom! Model fully trained, evaluated, and saved locally as: {model_filename}")

Evaluating model on the hidden test data...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.91      0.84      0.87    942296
           1       0.45      0.61      0.52    204290

    accuracy                           0.80   1146586
   macro avg       0.68      0.73      0.70   1146586
weighted avg       0.83      0.80      0.81   1146586

--- Confusion Matrix ---
[[789038 153258]
 [ 78748 125542]]

Boom! Model fully trained, evaluated, and saved locally as: xgboost_delay_model_v05_v01.joblib


In [7]:
from xgboost import XGBClassifier
import time

print("\nInitializing Tuned XGBoost Classifier...")
model = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    max_depth = 3,
    learning_rate = 0.01
)

print("Training the model. Please wait...")
start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

print(f"Success! Training complete in {round(end_time - start_time, 2)} seconds.")


Initializing Tuned XGBoost Classifier...
Training the model. Please wait...
Success! Training complete in 9.53 seconds.


In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Evaluating model on the hidden test data...")

# 1. Ask the model to make predictions
y_pred = model.predict(X_test)

# 2. Print out the KPI metrics
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 3. Export the trained model to your hard drive
model_filename = 'xgboost_delay_model_v05_v02.joblib'
joblib.dump(model, model_filename)

print(f"\nBoom! Model fully trained, evaluated, and saved locally as: {model_filename}")

Evaluating model on the hidden test data...

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.85      0.99      0.91    942296
           1       0.75      0.20      0.32    204290

    accuracy                           0.85   1146586
   macro avg       0.80      0.59      0.61   1146586
weighted avg       0.83      0.85      0.81   1146586

--- Confusion Matrix ---
[[928588  13708]
 [163535  40755]]

Boom! Model fully trained, evaluated, and saved locally as: xgboost_delay_model_v05_v02.joblib


In [9]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import time
import joblib

print("1. Calculating class imbalance for the safety net...")
imbalance_ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

print("2. Constructing the Master Parameter Grid...")
param_grid = {
    # --- 1. Tree Architecture (Complexity) ---
    'max_depth': [3, 4, 5, 6, 7],           # How deep to grow (7 is pushing it!)
    'gamma': [0, 1, 5],                     # Minimum loss reduction required to split a node
    
    # --- 2. Optimization Dynamics (Speed vs Stamina) ---
    'learning_rate': [0.01, 0.05, 0.1, 0.2],# Step size
    'n_estimators': [100, 200, 300, 500],   # Number of trees (Stamina)
    
    # --- 3. Randomization (Noise Protection) ---
    'subsample': [0.6, 0.8, 1.0],           # % of rows used per tree
    'colsample_bytree': [0.6, 0.8, 1.0],    # % of columns used per tree
    
    # --- 4. Regularization (Mathematical Penalties for Overfitting) ---
    'reg_alpha': [0, 0.1, 1],               # L1 penalty (encourages dropping useless features)
    'reg_lambda': [1, 10, 100]              # L2 penalty (prevents weights from getting too large)
}

print("3. Initializing the Base Engine...")
# We lock the random state for reproducibility and load our crucial imbalance ratio
xgb_base = XGBClassifier(
    random_state=42, 
    eval_metric='logloss', 
    scale_pos_weight=imbalance_ratio
)

print("4. Configuring the Randomized Search...")
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,         # Test 10 distinct, random combinations from the grid above
    scoring='f1',      # The compass: find the best balance of Precision and Recall
    cv=3,              # 3-Fold Cross Validation (Tests each combo 3 times on different data slices)
    verbose=2,         # Prints live progress so you know it hasn't frozen
    random_state=42,
    n_jobs=-1          # -1 tells Python to use 100% of your available CPU cores
)

print("5. Commencing Model Training (Grab a coffee, let the machine sweat)...")
start_time = time.time()

# The actual heavy lifting
random_search.fit(X_train, y_train)

end_time = time.time()
print(f"\nSuccess! Tuning complete in {round((end_time - start_time)/60, 2)} minutes.")

print("\n======================================")
print("🏆 THE CHAMPION PARAMETERS 🏆")
print("======================================")
for param, value in random_search.best_params_.items():
    print(f"{param}: {value}")

# Save the absolute best version of the model to your hard drive
best_model = random_search.best_estimator_
model_filename = 'xgboost_delay_model_ULTIMATE_v05.joblib'
joblib.dump(best_model, model_filename)

print(f"\nFinal model compiled and saved as: {model_filename}")

1. Calculating class imbalance for the safety net...
2. Constructing the Master Parameter Grid...
3. Initializing the Base Engine...
4. Configuring the Randomized Search...
5. Commencing Model Training (Grab a coffee, let the machine sweat)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Success! Tuning complete in 15.2 minutes.

🏆 THE CHAMPION PARAMETERS 🏆
subsample: 1.0
reg_lambda: 10
reg_alpha: 0
n_estimators: 300
max_depth: 6
learning_rate: 0.1
gamma: 0
colsample_bytree: 1.0

Final model compiled and saved as: xgboost_delay_model_ULTIMATE_v05.joblib


In [10]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix

print("Loading the Ultimate Champion Model from disk...")
# We load the .joblib file we saved in the last step
final_model = joblib.load('xgboost_delay_model_ULTIMATE_v05.joblib')

print("Evaluating the model on the hidden test data...")
# Ask the champion to make its predictions
y_pred_final = final_model.predict(X_test)

# Print the ultimate KPIs
print("\n🏆 --- FINAL CLASSIFICATION REPORT --- 🏆")
print(classification_report(y_test, y_pred_final))
g
print("🏆 --- FINAL CONFUSION MATRIX --- 🏆")
print(confusion_matrix(y_test, y_pred_final))

Loading the Ultimate Champion Model from disk...
Evaluating the model on the hidden test data...

🏆 --- FINAL CLASSIFICATION REPORT --- 🏆
              precision    recall  f1-score   support

           0       0.91      0.84      0.87    942296
           1       0.45      0.61      0.52    204290

    accuracy                           0.80   1146586
   macro avg       0.68      0.73      0.70   1146586
weighted avg       0.83      0.80      0.81   1146586

🏆 --- FINAL CONFUSION MATRIX --- 🏆
[[790490 151806]
 [ 78931 125359]]
